[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/01_basics.ipynb)

*Python from Scratch · Notebook 1 of 12*

# Python Basics

Your first programs: print, variables, data types, type conversion and input.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Think of a variable as a sticky label on a value, not a box. Most of Python gets easier once you see it that way.

**What interviewers really test here:** Types, integer vs float division, truthiness and the difference between printing and returning.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Your first program *(Basics)*
- [ ] 2. Variables: names for values *(Basics)*
- [ ] 3. Data types *(Basics)*
- [ ] 4. Converting between types, and input() *(Core)*
- [ ] 5. Printing nicely with f-strings *(Core)*
- [ ] 6. How Python stores values *(Advanced)*
- [ ] 7. Reading the practice answers *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### What is Python?

**Python** is a high-level, general-purpose programming language created by Guido van Rossum and first released in 1991. It is known for code that reads almost like English.



#### Key features




- **High-level** — you don't manage memory or hardware details.
- **Interpreted** — code runs line by line through the Python interpreter; no separate compile step for you.
- **Dynamically typed** — you don't declare types; each value carries its own type.
- **Strongly typed** — Python won't silently mix incompatible types (`"5" + 3` is an error).
- **Multi-paradigm** — supports procedural, object-oriented and functional styles.
- **Huge standard library** — "batteries included", plus hundreds of thousands of packages on PyPI.



#### Where it's used



Web development (Django, Flask, FastAPI), data analysis (pandas), machine learning and AI (PyTorch, scikit-learn), automation and scripting, testing, scientific computing.

### How Python runs your code

1. You write source code in a `.py` file (or type it into the interactive shell / REPL).
2. The interpreter (**CPython**, the standard one) compiles it to **bytecode** (`.pyc`), a simpler instruction format.
3. The **Python Virtual Machine** executes the bytecode one instruction at a time.



Errors come in two kinds: **syntax errors** are found before anything runs (e.g. a missing colon); **runtime errors (exceptions)** happen while running (e.g. dividing by zero).



Ways to run Python: the REPL (`python3` then type lines), a script (`python3 app.py`), notebooks (Jupyter), or an IDE such as VS Code or PyCharm.

### Syntax rules

- **Indentation is part of the syntax.** Blocks (inside `if`, loops, functions) are marked by indenting — 4 spaces by convention — not by braces.
- One statement per line normally. A line ends the statement; no semicolon needed.
- **Comments** start with `#`. Multi-line explanations at the top of a function use a **docstring** `"""…"""`.
- Python is **case-sensitive**: `print` works, `Print` doesn't.
- Long expressions can continue on the next line inside brackets `( )`, `[ ]`, `{ }`.

### Variables and identifiers

A **variable** is a name bound to an object in memory. It is created the first time you assign to it.



#### Identifier rules




- May contain letters, digits and underscores; must not start with a digit.
- Must not be a **keyword**: `False None True and as assert async await break class continue def del elif else except finally for from global if import in is lambda nonlocal not or pass raise return try while with yield`.
- Case-sensitive.



#### Conventions (PEP 8)



| Kind | Style | Example |
| --- | --- | --- |
| variables, functions | snake\_case | `total_price` |
| constants | UPPER\_CASE | `MAX_SIZE` |
| classes | PascalCase | `BankAccount` |
| "private" / internal | leading underscore | `_cache` |



**Multiple assignment**: `a, b = 1, 2`; **chained**: `x = y = 0`; **augmented**: `x += 1`.

### Built-in data types

<table> <tr><th>Category</th><th>Type</th><th>Example</th><th>Mutable?</th></tr> <tr><td rowspan="3">Numeric</td><td><code>int</code></td><td><code>42</code> (unlimited size)</td><td>no</td></tr> <tr><td><code>float</code></td><td><code>3.14</code> (64-bit decimal)</td><td>no</td></tr> <tr><td><code>complex</code></td><td><code>2 + 3j</code></td><td>no</td></tr> <tr><td>Text</td><td><code>str</code></td><td><code>"hello"</code></td><td>no</td></tr> <tr><td>Boolean</td><td><code>bool</code></td><td><code>True</code>, <code>False</code></td><td>no</td></tr> <tr><td>None</td><td><code>NoneType</code></td><td><code>None</code></td><td>no</td></tr> <tr><td rowspan="3">Sequence</td><td><code>list</code></td><td><code>[1, 2, 3]</code></td><td><b>yes</b></td></tr> <tr><td><code>tuple</code></td><td><code>(1, 2, 3)</code></td><td>no</td></tr> <tr><td><code>range</code></td><td><code>range(5)</code></td><td>no</td></tr> <tr><td>Mapping</td><td><code>dict</code></td><td><code>{"a": 1}</code></td><td><b>yes</b></td></tr> <tr><td rowspan="2">Set</td><td><code>set</code></td><td><code>{1, 2}</code></td><td><b>yes</b></td></tr> <tr><td><code>frozenset</code></td><td><code>frozenset({1, 2})</code></td><td>no</td></tr> </table>



Floats are stored in binary, so some decimals are approximate: `0.1 + 0.2` is `0.30000000000000004`. Use `round()` for display, or the `decimal` module for money.

### Type conversion

**Implicit conversion** (coercion) — Python converts automatically when it's safe: `int + float → float`, `True + 1 → 2`.



**Explicit conversion** (casting) — you call the type:



| Call | Result | Notes |
| --- | --- | --- |
| `int("42")` | 42 | text must be a whole number |
| `int(3.99)` | 3 | truncates toward zero |
| `float("2.5")` | 2.5 |   |
| `str(100)` | "100" | anything can become text |
| `bool(x)` | True/False | False for 0, 0.0, "", [], {}, set(), None |
| `list("abc")` | ['a', 'b', 'c'] | any iterable |
| `round(2.675, 2)` | 2.67 | banker's rounding and float precision |

### Input and output

- `print(*values, sep=" ", end="\n")` — writes values separated by `sep` and followed by `end`.
- `input(prompt)` — shows the prompt, waits for a line from the keyboard, returns it as a **str** without the newline.
- String formatting: f-strings `f"{x:.2f}"` (modern, preferred), `"{} {}".format(a, b)`, and the old `"%d" % n` style.



| Format spec | Meaning | Example → output |
| --- | --- | --- |
| `:.2f` | 2 decimal places | `f"{3.14159:.2f}"` → 3.14 |
| `:05` | pad to width 5 with zeros | `f"{42:05}"` → 00042 |
| `:,` | thousands separator | `f"{1234567:,}"` → 1,234,567 |
| `:>8` / `:<8` / `:^8` | right / left / centre in 8 chars | `f"{'hi':^8}"` → " hi " |
| `:%` | percentage | `f"{0.256:.1%}"` → 25.6% |

---
# Part 2: Lessons, step by step

## Lesson 1: Your first program

*Level: Basics*

A **program** is a list of instructions the computer follows **from top to bottom**, one line at a time. Python reads your code, and runs each line in order.



The first instruction everyone learns is `print`. It shows a value on the screen:



```python
print("Hello, world!")
print(2 + 3)
```



Text must be inside quotes (`"…"` or `'…'`) — that's a **string**. Numbers don't need quotes. Python works out `2 + 3` first and prints `5`.



**💡 Tip:** How to read the animations: the **highlighted line is the one about to run**. Press ▶ or the → key to run it. The black box shows everything printed so far.



Lines starting with `#` are **comments**. Python ignores them; they're notes for humans.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def first_program():
    # Python runs these lines from top to bottom
    print("Hello, world!")
    print(2 + 3)
    print("2 + 3")
    print("Python", "is", "fun")


first_program()

Hello, world!
5
2 + 3
Python is fun


## Lesson 2: Variables: names for values

*Level: Basics*

A **variable** is a name that points to a value, so you can use it later. You create one with `=`, the **assignment** operator:



```python
age = 20        # read it as: "age now refers to 20"
age = age + 1   # take the current value, add 1, store it back
```



`=` does **not** mean "equals" like in maths. It means "compute the right side, then make the name on the left point to the result". That's why `age = age + 1` makes sense.



Rules for names: letters, digits and underscores; can't start with a digit; case matters (`score` ≠ `Score`). Use clear names like `total_price`, not `tp`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def variables_demo():
    score = 10
    bonus = 5
    score = score + bonus
    score = score * 2
    name = "Mira"
    print(name, "scored", score)


variables_demo()

Mira scored 30


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 3: Data types

*Level: Basics*

Every value has a **type**, which decides what you can do with it.



| Type | Meaning | Examples |
| --- | --- | --- |
| `int` | whole numbers | `7`, `-3`, `1_000_000` |
| `float` | decimal numbers | `3.14`, `-0.5`, `2.0` |
| `str` | text | `"hi"`, `'42'`, `""` |
| `bool` | yes / no | `True`, `False` |
| `NoneType` | "no value" | `None` |



Ask Python for a type with `type(value)`. Note that `"42"` (a str) and `42` (an int) are different: `"42" + "1"` is `"421"`, while `42 + 1` is `43`.



Python is **dynamically typed**: a variable can refer to an int now and a str later. The *value* has a type; the name doesn't.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def types_demo():
    a = 42
    b = 3.5
    c = "42"
    d = True
    e = None
    print(type(a), type(b), type(c), type(d), type(e))
    print(a + 1, c + "1")


types_demo()

<class 'int'> <class 'float'> <class 'str'> <class 'bool'> <class 'NoneType'>
43 421


**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q10](#q10), [Q13](#q13). Try one now, or carry on and do them in Part 3.

## Lesson 4: Converting between types, and input()

*Level: Core*

You can convert values with the type's name used as a function:



```python
int("42")      # 42        text → whole number
float("3.5")   # 3.5
str(10)        # "10"      number → text
int(3.99)      # 3         cuts off the decimals (doesn't round)
bool(0)        # False     0, "", [], None are False; most else True
```



`input("question")` pauses the program, lets the user type, and returns what they typed — **always as a str**. So for numbers you convert:



```python
age = int(input("How old are you? "))
print("Next year you'll be", age + 1)
```



**⚠️ Watch out:** `int("3.5")` and `int("hello")` raise a `ValueError`. Later you'll learn to handle that with `try`/`except`.



Python also converts **automatically** in a few safe cases (**implicit conversion**): `3 + 2.5` gives the float `5.5`.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def conversion_demo():
    typed = "25"
    age = int(typed)
    next_year = age + 1
    price = float("19.99")
    label = "Age: " + str(age)
    print(label, "| next year:", next_year, "| price:", price)
    print(int(3.99), bool(0), bool("0"))


conversion_demo()

Age: 25 | next year: 26 | price: 19.99
3 False True


**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q17](#q17), [Q7](#q7), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 5: Printing nicely with f-strings

*Level: Core*

An **f-string** is a string with an `f` before the quote. Anything inside `{ }` is calculated and placed into the text:



```python
name, price, qty = "pen", 12.5, 3
print(f"{qty} {name}s cost {price * qty}")      # 3 pens cost 37.5
print(f"{price:.2f}")                            # 12.50  (2 decimals)
print(f"{7:03}")                                 # 007    (pad with zeros)
print(f"{'left':<8}|{'right':>8}")              # align in 8 characters
```



`print` also has two useful options: `sep` (what goes between values, default a space) and `end` (what goes at the end, default a new line).

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def fstring_demo():
    item = "notebook"
    price = 45.5
    qty = 3
    total = price * qty
    print(f"{qty} x {item} = {total}")
    print(f"Total with 2 decimals: {total:.2f}")
    print("a", "b", "c", sep="-")
    print("no new line", end="! ")
    print("done")


fstring_demo()

3 x notebook = 136.5
Total with 2 decimals: 136.50
a-b-c
no new line! done


**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q12](#q12), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 6: How Python stores values

*Level: Advanced*

In Python, every value is an **object** living in memory, and a variable is a **label** attached to an object. `x = 5` creates (or reuses) the object 5 and sticks the label `x` on it.




- `id(x)` gives the object's identity (like an address).
- `x is y` asks "same object?"; `x == y` asks "same value?".
- **Immutable** types (int, float, str, bool, tuple) can't be changed in place — "changing" them makes a new object.
- **Mutable** types (list, dict, set) can be changed in place, so two labels on the same list both see the change. You'll meet this again in the Lists topic.



This "labels on objects" model explains most surprises beginners hit later.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def labels_demo():
    a = 10
    b = a
    a = a + 1
    print("a =", a, "b =", b)
    x = [1, 2]
    y = x
    y.append(3)
    print("x =", x, "y =", y, "same object?", x is y)


labels_demo()

a = 11 b = 10
x = [1, 2, 3] y = [1, 2, 3] same object? True


## Lesson 7: Reading the practice answers

*Level: Advanced*

Each practice answer is written as a small **function**, so the site can test it automatically:



```python
def c_to_f(celsius):        # a recipe called c_to_f that takes one input
    return celsius * 9 / 5 + 32   # the answer it gives back
```



For now read `def name(inputs):` as "a recipe with these inputs" and `return` as "give back this answer". You'll learn functions properly in topic 9. In a real script you'd use it like this:



```python
temp = float(input("Celsius: "))
print(c_to_f(temp))
```



For every question you'll see two answers:




- **Brute force — the long way**: every step written out. Easy to follow; often longer or slower.
- **Optimal — the Pythonic way**: shorter, clearer or faster, using Python's built-in tools.



Both are shown running line by line. Try the question yourself first!

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [8]:
def program_demo():
    def c_to_f(celsius):
        return celsius * 9 / 5 + 32

    temps = [0, 37, 100]
    for t in temps:
        print(t, "C =", c_to_f(t), "F")


program_demo()

0 C = 32.0 F
37 C = 98.6 F
100 C = 212.0 F


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. c\_to\_f() returns 32.0
2. c\_to\_f() returns 98.6
3. c\_to\_f() returns 212.0
4. program\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q18](#q18), [Q20](#q20). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Swap Two Variables

🟢 Warm-up · *Variables · tuple assignment*

Given two values `a` and `b`, swap them and return `(a, b)`.

**Examples**

- `a = 3, b = 7` → `(7, 3)`
- `a = 'x', b = 'y'` → `('y', 'x')`

In [ ]:
def swap(a, b):
    """
    Given two values a and b, swap them and return (a, b).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(swap, [
    ((3, 7), (7, 3)),
    (('x', 'y'), ('y', 'x')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Use a third, temporary variable to hold one value while you overwrite it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Python can assign several names at once: `a, b = b, a` builds the pair (b, a) first, then unpacks it.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Use a third, temporary variable to hold one value while you overwrite it.

```python
def swap(a, b):
    temp = a
    a = b
    b = temp
    return a, b
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Python can assign several names at once: `a, b = b, a` builds the pair (b, a) first, then unpacks it.

```python
def swap(a, b):
    a, b = b, a
    return a, b
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Use a third, temporary variable to hold one value while you overwrite it. That costs O(1) time and O(1) space."

"I can do better: Python can assign several names at once: `a, b = b, a` builds the pair (b, a) first, then unpacks it. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/swap-two-variables)

<a id="q2"></a>
### Q2. Celsius to Fahrenheit

🟢 Warm-up · *Arithmetic · float*

Convert a temperature in Celsius to Fahrenheit using F = C × 9/5 + 32. Return the result rounded to 1 decimal place.

**Examples**

- `celsius = 100` → `212.0`
- `celsius = 0` → `32.0`
- `celsius = 36.6` → `97.9`

In [ ]:
def c_to_f(celsius):
    """
    Convert a temperature in Celsius to Fahrenheit using F = C × 9/5 + 32.
    Return the result rounded to 1 decimal place.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(c_to_f, [
    ((100,), 212.0),
    ((0,), 32.0),
    ((36.6,), 97.9),
    ((-40,), -40.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Do each part of the formula in its own variable so every step is visible.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Write the whole formula in one expression and round it.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Do each part of the formula in its own variable so every step is visible.

```python
def c_to_f(celsius):
    times = celsius * 9
    divided = times / 5
    fahrenheit = divided + 32
    result = round(fahrenheit, 1)
    return result
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Write the whole formula in one expression and round it.

```python
def c_to_f(celsius):
    return round(celsius * 9 / 5 + 32, 1)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Do each part of the formula in its own variable so every step is visible. That costs O(1) time and O(1) space."

"I can do better: Write the whole formula in one expression and round it. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/celsius-to-fahrenheit)

<a id="q3"></a>
### Q3. Rectangle Area and Perimeter

🟢 Warm-up · *Variables · returning two values*

Given the `length` and `width` of a rectangle, return `(area, perimeter)`.

**Examples**

- `length = 5, width = 3` → `(15, 16)`
- `length = 2.5, width = 4` → `(10.0, 13.0)`

In [ ]:
def rectangle(length, width):
    """
    Given the length and width of a rectangle, return (area, perimeter).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rectangle, [
    ((5, 3), (15, 16)),
    ((2.5, 4), (10.0, 13.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add up the four sides one by one for the perimeter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use the formulas area = l × w and perimeter = 2 × (l + w), and return both as a tuple.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Add up the four sides one by one for the perimeter.

```python
def rectangle(length, width):
    area = length * width
    perimeter = 0
    perimeter = perimeter + length
    perimeter = perimeter + width
    perimeter = perimeter + length
    perimeter = perimeter + width
    return area, perimeter
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use the formulas area = l × w and perimeter = 2 × (l + w), and return both as a tuple.

```python
def rectangle(length, width):
    return length * width, 2 * (length + width)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add up the four sides one by one for the perimeter. That costs O(1) time and O(1) space."

"I can do better: Use the formulas area = l × w and perimeter = 2 × (l + w), and return both as a tuple. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/rectangle-area-and-perimeter)

<a id="q4"></a>
### Q4. Add Two Numbers Given as Text

🟢 Warm-up · *Type conversion*

`a` and `b` are numbers written as strings, like `"12"`. Return their sum as an int.

**Examples**

- `a = '12', b = '30'` → `42`
- `a = '-5', b = '5'` → `0`
- `a = '007', b = '3'` → `10`

In [ ]:
def add_text(a, b):
    """
    a and b are numbers written as strings, like "12". Return their sum as an
    int.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(add_text, [
    (('12', '30'), 42),
    (('-5', '5'), 0),
    (('007', '3'), 10),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Convert each string digit by digit: multiply the running number by 10 and add the next digit.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`int()` converts a numeric string to an integer in one call.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Convert each string digit by digit: multiply the running number by 10 and add the next digit.

```python
def add_text(a, b):
    def to_int(s):
        sign = 1
        if s[0] == "-":
            sign = -1
            s = s[1:]
        number = 0
        for ch in s:
            number = number * 10 + (ord(ch) - ord("0"))
        return sign * number
    return to_int(a) + to_int(b)
```

**Cost:** O(d) — d digits time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `int()` converts a numeric string to an integer in one call.

```python
def add_text(a, b):
    return int(a) + int(b)
```

**Cost:** O(d) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Convert each string digit by digit: multiply the running number by 10 and add the next digit. That costs O(d) — d digits time and O(1) space."

"I can do better: `int()` converts a numeric string to an integer in one call. That's O(d) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/add-two-numbers-given-as-text)

<a id="q5"></a>
### Q5. Name of a Value's Type

🟢 Warm-up · *Data types · type()*

Return the name of the type of `value` as a string: `'int'`, `'float'`, `'str'`, `'bool'`, `'list'` or `'NoneType'`.

**Examples**

- `value = 5` → `'int'`
- `value = 2.5` → `'float'`
- `value = 'hi'` → `'str'`

In [ ]:
def type_name(value):
    """
    Return the name of the type of value as a string: 'int', 'float', 'str',
    'bool', 'list' or 'NoneType'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(type_name, [
    ((5,), 'int'),
    ((2.5,), 'float'),
    (('hi',), 'str'),
    ((True,), 'bool'),
    ((None,), 'NoneType'),
    (([1],), 'list'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each possible type with `isinstance`, one after another. Check bool before int, because True is also an int.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`type(value)` gives the class; its `__name__` is the name as text.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check each possible type with `isinstance`, one after another. Check bool before int, because True is also an int.

```python
def type_name(value):
    if value is None:
        return "NoneType"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, int):
        return "int"
    if isinstance(value, float):
        return "float"
    if isinstance(value, str):
        return "str"
    if isinstance(value, list):
        return "list"
    return "unknown"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `type(value)` gives the class; its `__name__` is the name as text.

```python
def type_name(value):
    return type(value).__name__
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each possible type with `isinstance`, one after another. Check bool before int, because True is also an int. That costs O(1) time and O(1) space."

"I can do better: `type(value)` gives the class; its `__name__` is the name as text. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/name-of-a-values-type)

<a id="q6"></a>
### Q6. Minutes to Hours and Minutes

🟢 Warm-up · *// and %*

Convert a number of `minutes` into `(hours, minutes_left)`.

**Examples**

- `minutes = 135` → `(2, 15)`
- `minutes = 59` → `(0, 59)`
- `minutes = 120` → `(2, 0)`

In [ ]:
def to_hours(minutes):
    """
    Convert a number of minutes into (hours, minutes_left).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(to_hours, [
    ((135,), (2, 15)),
    ((59,), (0, 59)),
    ((120,), (2, 0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Take away 60 minutes at a time, counting the hours.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Floor division `//` gives the whole hours and modulo `%` gives what's left; `divmod` does both.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Take away 60 minutes at a time, counting the hours.

```python
def to_hours(minutes):
    hours = 0
    while minutes >= 60:
        minutes = minutes - 60
        hours = hours + 1
    return hours, minutes
```

**Cost:** O(minutes / 60) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Floor division `//` gives the whole hours and modulo `%` gives what's left; `divmod` does both.

```python
def to_hours(minutes):
    return divmod(minutes, 60)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Take away 60 minutes at a time, counting the hours. That costs O(minutes / 60) time and O(1) space."

"I can do better: Floor division `//` gives the whole hours and modulo `%` gives what's left; `divmod` does both. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/minutes-to-hours-and-minutes)

<a id="q7"></a>
### Q7. Sum of Digits

🟢 Warm-up · *// and % · str*

Return the sum of the digits of a non-negative integer `n`.

**Examples**

- `n = 123` → `6`
- `n = 9045` → `18`
- `n = 0` → `0`

In [ ]:
def digit_sum(n):
    """
    Return the sum of the digits of a non-negative integer n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(digit_sum, [
    ((123,), 6),
    ((9045,), 18),
    ((0,), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Repeatedly take the last digit with `% 10` and remove it with `// 10`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Turn the number into a string, convert each character back to int, and add them with `sum`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Repeatedly take the last digit with `% 10` and remove it with `// 10`.

```python
def digit_sum(n):
    total = 0
    while n > 0:
        last = n % 10
        total = total + last
        n = n // 10
    return total
```

**Cost:** O(d) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Turn the number into a string, convert each character back to int, and add them with `sum`.

```python
def digit_sum(n):
    return sum(int(d) for d in str(n))
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Repeatedly take the last digit with `% 10` and remove it with `// 10`. That costs O(d) time and O(1) space."

"I can do better: Turn the number into a string, convert each character back to int, and add them with `sum`. That's O(d) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/sum-of-digits)

<a id="q8"></a>
### Q8. Last Digit

🟢 Warm-up · *% operator*

Return the last digit of an integer (ignore the sign).

**Examples**

- `n = 1234` → `4`
- `n = -87` → `7`
- `n = 5` → `5`

In [ ]:
def last_digit(n):
    """
    Return the last digit of an integer (ignore the sign).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(last_digit, [
    ((1234,), 4),
    ((-87,), 7),
    ((5,), 5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Convert to a string, drop a leading minus, and read the last character.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`abs(n) % 10` is the remainder after dividing by 10 — exactly the last digit.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Convert to a string, drop a leading minus, and read the last character.

```python
def last_digit(n):
    text = str(n)
    if text[0] == "-":
        text = text[1:]
    last_char = text[len(text) - 1]
    return int(last_char)
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `abs(n) % 10` is the remainder after dividing by 10 — exactly the last digit.

```python
def last_digit(n):
    return abs(n) % 10
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Convert to a string, drop a leading minus, and read the last character. That costs O(d) time and O(d) space."

"I can do better: `abs(n) % 10` is the remainder after dividing by 10 — exactly the last digit. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/last-digit)

<a id="q9"></a>
### Q9. Average of Three Numbers

🟢 Warm-up · *Arithmetic*

Return the average of `a`, `b` and `c`, rounded to 2 decimals.

**Examples**

- `a = 1, b = 2, c = 3` → `2.0`
- `a = 10, b = 20, c = 25` → `18.33`
- `a = -1, b = 0, c = 1` → `0.0`

In [ ]:
def average3(a, b, c):
    """
    Return the average of a, b and c, rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(average3, [
    ((1, 2, 3), 2.0),
    ((10, 20, 25), 18.33),
    ((-1, 0, 1), 0.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Add the numbers one at a time into a total, count them, then divide.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Put them in a list and use `sum(nums) / len(nums)` — the same code works for any amount of numbers.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Add the numbers one at a time into a total, count them, then divide.

```python
def average3(a, b, c):
    total = 0
    total = total + a
    total = total + b
    total = total + c
    count = 3
    avg = total / count
    return round(avg, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Put them in a list and use `sum(nums) / len(nums)` — the same code works for any amount of numbers.

```python
def average3(a, b, c):
    nums = [a, b, c]
    return round(sum(nums) / len(nums), 2)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Add the numbers one at a time into a total, count them, then divide. That costs O(1) time and O(1) space."

"I can do better: Put them in a list and use `sum(nums) / len(nums)` — the same code works for any amount of numbers. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/average-of-three-numbers)

<a id="q10"></a>
### Q10. Is It Truthy?

🟢 Warm-up · *bool · truthiness*

Return `True` if `value` counts as true in an `if`, else `False`. Empty things, 0 and None are false.

**Examples**

- `value = 0` → `False`
- `value = 7` → `True`
- `value = ''` → `False`

In [ ]:
def is_truthy(value):
    """
    Return True if value counts as true in an if, else False. Empty things, 0
    and None are false.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_truthy, [
    ((0,), False),
    ((7,), True),
    (('',), False),
    (('0',), True),
    (([],), False),
    ((None,), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each kind of 'empty' value by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`bool(value)` applies Python's truthiness rules directly.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check each kind of 'empty' value by hand.

```python
def is_truthy(value):
    if value is None:
        return False
    if value == 0:
        return False
    if value == "" or value == [] or value == {} or value == ():
        return False
    return True
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `bool(value)` applies Python's truthiness rules directly.

```python
def is_truthy(value):
    return bool(value)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each kind of 'empty' value by hand. That costs O(1) time and O(1) space."

"I can do better: `bool(value)` applies Python's truthiness rules directly. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/is-it-truthy)

<a id="q11"></a>
### Q11. Full Name

🟢 Warm-up · *String joining · f-strings*

Return the full name made of `first` and `last` with one space between, e.g. `'Ada Lovelace'`.

**Examples**

- `first = 'Ada', last = 'Lovelace'` → `'Ada Lovelace'`
- `first = 'Ravi', last = 'K'` → `'Ravi K'`

In [ ]:
def full_name(first, last):
    """
    Return the full name made of first and last with one space between, e.g.
    'Ada Lovelace'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(full_name, [
    (('Ada', 'Lovelace'), 'Ada Lovelace'),
    (('Ravi', 'K'), 'Ravi K'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Glue the pieces together with `+`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

An f-string puts values straight into the text: `f"{first} {last}"`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Glue the pieces together with `+`.

```python
def full_name(first, last):
    result = first
    result = result + " "
    result = result + last
    return result
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** An f-string puts values straight into the text: `f"{first} {last}"`.

```python
def full_name(first, last):
    return f"{first} {last}"
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Glue the pieces together with `+`. That costs O(n) time and O(n) space."

"I can do better: An f-string puts values straight into the text: `f"{first} {last}"`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/full-name)

<a id="q12"></a>
### Q12. Format Money

🟢 Warm-up · *Format specifiers*

Return `amount` as text with exactly 2 decimal places, e.g. `5 → '5.00'`, `3.14159 → '3.14'`.

**Examples**

- `amount = 5` → `'5.00'`
- `amount = 3.14159` → `'3.14'`
- `amount = 2.5` → `'2.50'`

In [ ]:
def money(amount):
    """
    Return amount as text with exactly 2 decimal places, e.g. 5 → '5.00',
    3.14159 → '3.14'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(money, [
    ((5,), '5.00'),
    ((3.14159,), '3.14'),
    ((2.5,), '2.50'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Round, convert to text, then add missing zeros after the dot by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The format spec `:.2f` means 'float with 2 decimals'.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Round, convert to text, then add missing zeros after the dot by hand.

```python
def money(amount):
    text = str(round(float(amount), 2))
    whole, dot, frac = text.partition(".")
    while len(frac) < 2:
        frac = frac + "0"
    return whole + "." + frac
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** The format spec `:.2f` means 'float with 2 decimals'.

```python
def money(amount):
    return f"{amount:.2f}"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Round, convert to text, then add missing zeros after the dot by hand. That costs O(1) time and O(1) space."

"I can do better: The format spec `:.2f` means 'float with 2 decimals'. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/format-money)

<a id="q13"></a>
### Q13. Count the True Values

🟢 Warm-up · *bool is a kind of int*

Given three booleans, return how many are `True`.

**Examples**

- `a = True, b = False, c = True` → `2`
- `a = False, b = False, c = False` → `0`
- `a = True, b = True, c = True` → `3`

In [ ]:
def count_true(a, b, c):
    """
    Given three booleans, return how many are True.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_true, [
    ((True, False, True), 2),
    ((False, False, False), 0),
    ((True, True, True), 3),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check each one with `if` and add 1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

True behaves like 1 and False like 0, so `sum` counts them.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check each one with `if` and add 1.

```python
def count_true(a, b, c):
    count = 0
    if a:
        count = count + 1
    if b:
        count = count + 1
    if c:
        count = count + 1
    return count
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** True behaves like 1 and False like 0, so `sum` counts them.

```python
def count_true(a, b, c):
    return sum([a, b, c])
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check each one with `if` and add 1. That costs O(1) time and O(1) space."

"I can do better: True behaves like 1 and False like 0, so `sum` counts them. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/count-the-true-values)

<a id="q14"></a>
### Q14. Distance Between Two Points

🟢 Warm-up · *math module*

Return the straight-line distance between points (x1, y1) and (x2, y2), rounded to 2 decimals.

**Examples**

- `x1 = 0, y1 = 0, x2 = 3, y2 = 4` → `5.0`
- `x1 = 1, y1 = 1, x2 = 4, y2 = 5` → `5.0`
- `x1 = -1, y1 = 2, x2 = 2, y2 = -2` → `5.0`

In [ ]:
def distance(x1, y1, x2, y2):
    """
    Return the straight-line distance between points (x1, y1) and (x2, y2),
    rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(distance, [
    ((0, 0, 3, 4), 5.0),
    ((1, 1, 4, 5), 5.0),
    ((-1, 2, 2, -2), 5.0),
    ((0, 0, 1, 1), 1.41),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute the differences, square them, add, and take the square root with `** 0.5`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.dist` (or `math.hypot`) does the Pythagoras formula for you.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compute the differences, square them, add, and take the square root with `** 0.5`.

```python
def distance(x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1
    square_sum = dx * dx + dy * dy
    root = square_sum ** 0.5
    return round(root, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `math.dist` (or `math.hypot`) does the Pythagoras formula for you.

```python
def distance(x1, y1, x2, y2):
    import math
    return round(math.dist((x1, y1), (x2, y2)), 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute the differences, square them, add, and take the square root with `** 0.5`. That costs O(1) time and O(1) space."

"I can do better: `math.dist` (or `math.hypot`) does the Pythagoras formula for you. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/distance-between-two-points)

<a id="q15"></a>
### Q15. Quotient and Remainder

🟢 Warm-up · *divmod*

For positive integers `a` and `b`, return `(a // b, a % b)` without using `//` or `%` in the long way.

**Examples**

- `a = 17, b = 5` → `(3, 2)`
- `a = 20, b = 4` → `(5, 0)`
- `a = 3, b = 7` → `(0, 3)`

In [ ]:
def quot_rem(a, b):
    """
    For positive integers a and b, return (a // b, a % b) without using // or %
    in the long way.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(quot_rem, [
    ((17, 5), (3, 2)),
    ((20, 4), (5, 0)),
    ((3, 7), (0, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Subtract b from a repeatedly, counting how many times it fits.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`divmod(a, b)` returns both at once.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Subtract b from a repeatedly, counting how many times it fits.

```python
def quot_rem(a, b):
    count = 0
    while a >= b:
        a = a - b
        count = count + 1
    return count, a
```

**Cost:** O(a / b) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `divmod(a, b)` returns both at once.

```python
def quot_rem(a, b):
    return divmod(a, b)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Subtract b from a repeatedly, counting how many times it fits. That costs O(a / b) time and O(1) space."

"I can do better: `divmod(a, b)` returns both at once. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/quotient-and-remainder)

<a id="q16"></a>
### Q16. Kilometres to Miles

🟢 Warm-up · *Constants · rounding*

Convert kilometres to miles (1 km = 0.621371 miles), rounded to 2 decimals.

**Examples**

- `km = 10` → `6.21`
- `km = 42.195` → `26.22`
- `km = 0` → `0.0`

In [ ]:
def km_to_miles(km):
    """
    Convert kilometres to miles (1 km = 0.621371 miles), rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(km_to_miles, [
    ((10,), 6.21),
    ((42.195,), 26.22),
    ((0,), 0.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Store the factor in a variable, multiply, then round in a separate step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use a named constant in capitals and one expression.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Store the factor in a variable, multiply, then round in a separate step.

```python
def km_to_miles(km):
    factor = 0.621371
    miles = km * factor
    rounded = round(miles, 2)
    return rounded
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use a named constant in capitals and one expression.

```python
def km_to_miles(km):
    MILES_PER_KM = 0.621371
    return round(km * MILES_PER_KM, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Store the factor in a variable, multiply, then round in a separate step. That costs O(1) time and O(1) space."

"I can do better: Use a named constant in capitals and one expression. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/kilometres-to-miles)

<a id="q17"></a>
### Q17. Is It a Whole Number String?

🟢 Warm-up · *str methods · conversion*

Return `True` if the text is a whole number made only of digits (like `'2024'`), else `False`.

**Examples**

- `text = '2024'` → `True`
- `text = '12a'` → `False`
- `text = ''` → `False`

In [ ]:
def is_whole_number(text):
    """
    Return True if the text is a whole number made only of digits (like '2024'),
    else False.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_whole_number, [
    (('2024',), True),
    (('12a',), False),
    (('',), False),
    (('3.5',), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every character is between '0' and '9', and that the text isn't empty.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`str.isdigit()` does exactly this check (it's False for an empty string).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check every character is between '0' and '9', and that the text isn't empty.

```python
def is_whole_number(text):
    if text == "":
        return False
    for ch in text:
        if ch < "0" or ch > "9":
            return False
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `str.isdigit()` does exactly this check (it's False for an empty string).

```python
def is_whole_number(text):
    return text.isdigit()
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every character is between '0' and '9', and that the text isn't empty. That costs O(n) time and O(1) space."

"I can do better: `str.isdigit()` does exactly this check (it's False for an empty string). That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/is-it-a-whole-number-string)

<a id="q18"></a>
### Q18. Greeting From Input

🟢 Warm-up · *input() · f-strings*

A program reads a name and an age with `input()`. Write the function that builds the reply: given `name` and `age_text` (both strings), return `'Hi NAME, next year you will be AGE+1'`.

**Examples**

- `name = 'Asha', age_text = '20'` → `'Hi Asha, next year you will be 21'`
- `name = 'Li', age_text = '9'` → `'Hi Li, next year you will be 10'`

In [ ]:
def greet(name, age_text):
    """
    A program reads a name and an age with input(). Write the function that
    builds the reply: given name and age_text (both strings), return 'Hi NAME,
    next year you will be AGE+1'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(greet, [
    (('Asha', '20'), 'Hi Asha, next year you will be 21'),
    (('Li', '9'), 'Hi Li, next year you will be 10'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Convert the age, add one, convert back to text, and join the pieces with `+`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

An f-string can contain the calculation directly.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Convert the age, add one, convert back to text, and join the pieces with `+`.

```python
def greet(name, age_text):
    age = int(age_text)
    next_age = age + 1
    next_age_text = str(next_age)
    return "Hi " + name + ", next year you will be " + next_age_text
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** An f-string can contain the calculation directly.

```python
def greet(name, age_text):
    return f"Hi {name}, next year you will be {int(age_text) + 1}"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Convert the age, add one, convert back to text, and join the pieces with `+`. That costs O(1) time and O(1) space."

"I can do better: An f-string can contain the calculation directly. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/greeting-from-input)

<a id="q19"></a>
### Q19. Seconds to Clock Time

🟡 Interview standard · *divmod · formatting*

Convert `seconds` into a string `'HH:MM:SS'` with two digits each.

**Examples**

- `seconds = 3725` → `'01:02:05'`
- `seconds = 59` → `'00:00:59'`
- `seconds = 86399` → `'23:59:59'`

In [ ]:
def clock(seconds):
    """
    Convert seconds into a string 'HH:MM:SS' with two digits each.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(clock, [
    ((3725,), '01:02:05'),
    ((59,), '00:00:59'),
    ((86399,), '23:59:59'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute hours, minutes and seconds step by step and add a leading zero by hand when a part is below 10.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use `divmod` twice and the format spec `:02` to pad with zeros.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compute hours, minutes and seconds step by step and add a leading zero by hand when a part is below 10.

```python
def clock(seconds):
    hours = seconds // 3600
    seconds = seconds - hours * 3600
    minutes = seconds // 60
    seconds = seconds - minutes * 60
    parts = []
    for part in [hours, minutes, seconds]:
        if part < 10:
            parts.append("0" + str(part))
        else:
            parts.append(str(part))
    return parts[0] + ":" + parts[1] + ":" + parts[2]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use `divmod` twice and the format spec `:02` to pad with zeros.

```python
def clock(seconds):
    minutes, s = divmod(seconds, 60)
    h, m = divmod(minutes, 60)
    return f"{h:02}:{m:02}:{s:02}"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute hours, minutes and seconds step by step and add a leading zero by hand when a part is below 10. That costs O(1) time and O(1) space."

"I can do better: Use `divmod` twice and the format spec `:02` to pad with zeros. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/seconds-to-clock-time)

<a id="q20"></a>
### Q20. Bill Split With Tip

🟡 Interview standard · *Expressions · rounding*

A bill of `total` is shared by `people` with a `tip_percent` tip. Return each person's share rounded to 2 decimals.

**Examples**

- `total = 100, people = 4, tip_percent = 10` → `27.5`
- `total = 59.99, people = 3, tip_percent = 15` → `23.0`
- `total = 80, people = 2, tip_percent = 0` → `40.0`

In [ ]:
def split_bill(total, people, tip_percent):
    """
    A bill of total is shared by people with a tip_percent tip. Return each
    person's share rounded to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(split_bill, [
    ((100, 4, 10), 27.5),
    ((59.99, 3, 15), 23.0),
    ((80, 2, 0), 40.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute the tip, the new total and the share in separate steps.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One expression: total × (1 + tip/100) ÷ people.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compute the tip, the new total and the share in separate steps.

```python
def split_bill(total, people, tip_percent):
    tip = total * tip_percent / 100
    with_tip = total + tip
    share = with_tip / people
    return round(share, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** One expression: total × (1 + tip/100) ÷ people.

```python
def split_bill(total, people, tip_percent):
    return round(total * (1 + tip_percent / 100) / people, 2)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute the tip, the new total and the share in separate steps. That costs O(1) time and O(1) space."

"I can do better: One expression: total × (1 + tip/100) ÷ people. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/basics/bill-split-with-tip)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Is Python compiled or interpreted?</b></summary>

Both, technically. CPython compiles source to bytecode, then interprets that bytecode in a virtual machine. Because the compilation is automatic and hidden, Python is usually called an interpreted language.

</details>

<details>
<summary><b>2. What does 'dynamically typed' mean?</b></summary>

Types belong to values, not to variable names, and are checked while the program runs. The same name can refer to an int and later a string. In statically typed languages (Java, C) you declare a variable's type up front.

</details>

<details>
<summary><b>3. What is the difference between = and ==?</b></summary>

`=` assigns: it binds a name to a value. `==` compares two values and gives True or False.

</details>

<details>
<summary><b>4. What is the difference between is and ==?</b></summary>

`==` checks equal values; `is` checks that both names refer to the very same object. Use `is` only for singletons like `None` (`x is None`).

</details>

<details>
<summary><b>5. Mutable vs immutable — give examples.</b></summary>

Mutable objects can be changed in place: list, dict, set. Immutable objects can't: int, float, str, bool, tuple, frozenset. 'Changing' an immutable value really creates a new object.

</details>

<details>
<summary><b>6. Why does 0.1 + 0.2 not equal 0.3?</b></summary>

Floats are stored in binary, and 0.1 has no exact binary representation, so tiny rounding errors appear. Compare floats with `math.isclose()` and use `decimal.Decimal` for exact money calculations.

</details>

<details>
<summary><b>7. What type does input() return?</b></summary>

Always `str`. Convert it with `int()` or `float()` when you need a number.

</details>

<details>
<summary><b>8. What is None?</b></summary>

A special value of type NoneType meaning 'no value'. Functions without a `return` give back None. Test for it with `is None`.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Your first program
- [ ] I can explain: Variables: names for values
- [ ] I can explain: Data types
- [ ] I can explain: Converting between types, and input()
- [ ] I can explain: Printing nicely with f-strings
- [ ] I can explain: How Python stores values
- [ ] I can explain: Reading the practice answers

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q19](#q19) Seconds to Clock Time
- [ ] [Q20](#q20) Bill Split With Tip

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.